# Cati 01 — 토크나이저 학습

**가속기: 없음 (CPU)** ← 오른쪽 Settings에서 Accelerator를 **None**으로 두세요.
CPU 세션은 TPU 쿼터를 쓰지 않습니다. 20시간을 아끼는 겁니다.

## 이 노트북이 하는 일
1. FineWeb2 한국어 스트리밍이 실제로 되는지 확인
2. 끊고 이어받기가 정확한지 확인
3. 토크나이저 학습 (vocab 49,152)
4. 한국어 압축률 측정 — 기성 토크나이저와 비교

## 왜 중요한가
계산량은 **토큰** 단위로 붙습니다. 한국어 압축률이 0.47(SmolLM2급)에서 2.5로
올라가면 같은 GPU 시간에 **한국어 원문이 5.3배** 들어갑니다.

## ⚠️ 한 번만 하고 다시는 바꾸지 않습니다
전 티어(50M/100M/350M)가 같은 토크나이저를 공유해야 실험 결과를 비교할 수 있습니다.

## 끝나고 할 일
`Save Version` → 다음 노트북에서 이 노트북의 **Output**을 Input으로 추가합니다.

In [ ]:
# ── 코드 가져오기 ────────────────────────────────────────────────
# 코드를 고쳤으면 GitHub에 push 한 뒤 이 노트북을 다시 실행하면 된다.
GITHUB_URL = "https://github.com/foeplob11-code/Cati.git"
DATASET_DIR = "/kaggle/input/cati-code"   # GitHub 대신 Dataset으로 올린 경우

import os, subprocess, sys, shutil
from pathlib import Path

if GITHUB_URL:
    if not Path("/kaggle/working/Cati").exists():
        subprocess.run(["git", "clone", "--depth", "1", GITHUB_URL,
                        "/kaggle/working/Cati"], check=True)
    CATI = Path("/kaggle/working/Cati")
elif Path(DATASET_DIR).exists():
    # Dataset은 읽기 전용이라 작업 디렉터리로 복사한다
    CATI = Path("/kaggle/working/Cati")
    if not CATI.exists():
        shutil.copytree(DATASET_DIR, CATI)
else:
    raise SystemExit(
        "코드를 못 찾았다. 둘 중 하나를 하세요:\n"
        "  (a) GitHub에 올리고 위 GITHUB_URL 채우기\n"
        "  (b) Cati 폴더를 Kaggle Dataset 'cati-code'로 업로드하고 이 노트북에 추가하기")

os.chdir(CATI)
sys.path.insert(0, str(CATI))
print("코드:", CATI)
print("파일:", sorted(p.name for p in CATI.iterdir() if not p.name.startswith(".")))

In [ ]:
# ── 패키지 ───────────────────────────────────────────────────────
!pip install -q "tokenizers>=0.22" "datasets>=3.0" 2>&1 | tail -2
import tokenizers, datasets
print("tokenizers", tokenizers.__version__, "· datasets", datasets.__version__)

## 1. 인터넷 확인

Settings → Internet 을 **On** 으로 해야 합니다.

In [ ]:
import socket
try:
    socket.create_connection(("huggingface.co", 443), timeout=10).close()
    print("인터넷 OK")
except OSError as e:
    raise SystemExit(f"인터넷이 꺼져 있다: {e}\n"
                     "오른쪽 Settings → Internet → On 으로 켜고 다시 실행")

## 2. 스트리밍 + 이어받기 검증

여기가 유일하게 검증 안 된 부분이었습니다. `datasets` 의 네이티브 재개가
동작하지 않으면 세션마다 앞부분을 다시 읽어야 해서 5주 런에서 비용이 커집니다.

In [ ]:
from cati.stream import HFSource, ResumableStream

src = HFSource("ko", "HuggingFaceFW/fineweb-2", "kor_Hang", "text")
st = ResumableStream([src], [1.0], seed=0)
it = iter(st)

print("연결 중...")
head = [next(it)[1] for _ in range(4)]
print(f"네이티브 재개 지원: {src._native}")
for i, t in enumerate(head):
    print(f"  {i}: {t[:70].replace(chr(10), ' ')}...")

state = st.state_dict()
after = [next(it)[1][:60] for _ in range(3)]

st2 = ResumableStream([HFSource("ko", "HuggingFaceFW/fineweb-2", "kor_Hang", "text")],
                      [1.0], seed=0)
st2.load_state_dict(state)
resumed = [next(iter(st2))[1][:60] for _ in range(1)]

ok = after[0] == resumed[0]
print(f"\n이어받기: {'정확 ✅' if ok else '어긋남 ⚠️ — 건너뛰기 경로로 동작한다'}")
if not ok:
    print("  → 치명적이지는 않지만, 사전 토큰화 방식 전환을 검토할 것")

## 3. 토크나이저 학습

문서 40만 개로 학습합니다. vocab 49,152에는 충분합니다.
RAM이 남으면 `--docs` 를 올려도 되지만 효과는 크지 않습니다.

10~30분 걸립니다.

In [ ]:
!python scripts/train_tokenizer.py train --docs 400000

## 4. 압축률 측정 — 목표를 넘겼는지

In [ ]:
!pip install -q transformers 2>&1 | tail -1
!python scripts/train_tokenizer.py measure --baseline Qwen/Qwen3-1.7B

## 5. 결과 저장

`/kaggle/working` 에 두면 `Save Version` 할 때 노트북 Output으로 보존됩니다.
다음 노트북에서 이걸 Input으로 붙입니다.

In [ ]:
import shutil
from pathlib import Path

src = Path("artifacts/tokenizer/tokenizer.json")
dst = Path("/kaggle/working/tokenizer.json")
assert src.exists(), "토크나이저가 만들어지지 않았다 — 위 셀 출력을 확인할 것"
shutil.copy(src, dst)
print(f"저장: {dst}  ({dst.stat().st_size/1e6:.2f} MB)")
print("\n다음: 우측 상단 Save Version → Save & Run All")
print("     그 다음 02_train 노트북에서 이 노트북의 Output을 Input으로 추가")